In [1]:
import json
import os
import sys
import time

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED

from topnum.regularizers import (
    FastFixPhiRegularizer, DecorrelateWithOtherPhiRegularizer, DecorrelateWithOtherPhiRegularizer2
)
from topnum.scores.intratext_coherence_score import (
    IntratextCoherenceScore,
    ComputationMethod,
    WordTopicRelatednessType,
)

In [4]:
import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

In [5]:
topicnet.__file__

! ls /home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [6]:
DATA_FOLDER_PATH = '/data_mil/shared/CompressaAI/iterative/data/noow'

In [7]:
! ls $DATA_FOLDER_PATH

_20_Newsgroups.csv	       MKB_10_NOOW__internals
_20_Newsgroups__internals      Post_Science__internals
20_Newsgroups_NOOW.csv	       Post_Science_NOOW.csv
20_Newsgroups_NOOW__internals  Post_Science_NOOW_fixed.csv
_Lenta.csv		       Post_Science_NOOW_fixed__internals
MKB_10__internals	       Post_Science_NOOW__internals
MKB_10_NOOW.csv		       WikiRef_220_NOOW.csv


In [8]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/20_Newsgroups_NOOW.csv',
)

dataset.get_possible_modalities()

{'@lemmatized'}

In [9]:
dataset.get_dictionary()

artm.Dictionary(name=7e8096b9-afd2-47f1-a9c6-3f0f5fbbc228, num_entries=52744)

In [10]:
dataset._internals_folder_path

'/data_mil/shared/CompressaAI/iterative/data/noow/20_Newsgroups_NOOW__internals'

In [11]:
! ls '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/'

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [12]:
MAIN_MODALITY = '@lemmatized'

In [13]:
dataset._data.head()

,raw_text,id,vw_text
id,,,
rec_autos_102994,I was wondering if anyone out there could enli...,rec_autos_102994,rec_autos_102994 |@lemmatized wonder anyone co...
comp_sys_mac_hardware_51861,A fair number of brave souls who upgraded thei...,comp_sys_mac_hardware_51861,comp_sys_mac_hardware_51861 |@lemmatized fair ...
comp_sys_mac_hardware_51879,"well folks, my mac plus finally gave up the gh...",comp_sys_mac_hardware_51879,comp_sys_mac_hardware_51879 |@lemmatized well ...
comp_graphics_38242,\nDo you have Weitek's address/phone number? ...,comp_graphics_38242,comp_graphics_38242 |@lemmatized weitek addres...
sci_space_60880,"From article <C5owCB.n3p@world.std.com>, by to...",sci_space_60880,sci_space_60880 |@lemmatized article tom baker...


In [14]:
dictionary = dataset.get_dictionary()

print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

print(dictionary)

artm.Dictionary(name=7e8096b9-afd2-47f1-a9c6-3f0f5fbbc228, num_entries=52744)
artm.Dictionary(name=7e8096b9-afd2-47f1-a9c6-3f0f5fbbc228, num_entries=52744)


In [15]:
dataset._cached_dict = dictionary

In [16]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [17]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 3.43 s, sys: 179 ms, total: 3.61 s
Wall time: 3.56 s


In [18]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [19]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [18]:
KnownModel

<enum 'KnownModel'>

In [19]:
PARAMS_EXPLORED

{<KnownModel.LDA: 'LDA'>: {'prior': ['symmetric', 'asymmetric', 'heuristic']},
 <KnownModel.PLSA: 'PLSA'>: {},
 <KnownModel.TLESS: 'TARTM'>: {},
 <KnownModel.SPARSE: 'sparse'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1]},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': [0.02,
   0.05,
   0.1]},
 <KnownModel.ARTM: 'ARTM'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1],
  'decorrelation_tau': [0.02, 0.05, 0.1]}}

In [20]:
NUM_TOPICS = 50  # vary
NUM_TRAINS = 3
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

## Test

In [20]:
PARAMS_EXPLORED[KnownModel.PLSA]


model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=1,
)

model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [21]:
list(model.scores.keys())

['PerplexityScore@all',
 'SparsityThetaScore',
 'SparsityPhiScore@lemmatized',
 'PerplexityScore@lemmatized',
 'TopicKernel@lemmatized.average_coherence',
 'TopicKernel@lemmatized.average_contrast',
 'TopicKernel@lemmatized.average_purity',
 'TopicKernel@lemmatized.average_size',
 'TopicKernel@lemmatized.coherence',
 'TopicKernel@lemmatized.contrast',
 'TopicKernel@lemmatized.purity',
 'TopicKernel@lemmatized.size',
 'TopicKernel@lemmatized.tokens']

In [22]:
model.scores[f'PerplexityScore{MAIN_MODALITY}']

[50603.0234375,
 4320.7158203125,
 3532.169921875,
 2861.91650390625,
 2524.459716796875,
 2342.09423828125,
 2236.879638671875,
 2172.470458984375,
 2130.379638671875,
 2100.591796875,
 2078.34765625,
 2061.140380859375,
 2047.145751953125,
 2036.034912109375,
 2026.8226318359375,
 2018.91015625,
 2012.030029296875,
 2005.9998779296875,
 2000.5599365234375,
 1995.385986328125]

In [23]:
phi = model.get_phi()
target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
target_topic_names = [phi.columns[i] for i in target_topic_indices]

custom_scores = [
    TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )
    for top in [20]  # [10, 20, 50, 100]
]
custom_scores = custom_scores + [
    DiversityScore(
        name=f'diversity_{metric}',
        topic_names=['topic_0', 'topic_1'],
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

for score in custom_scores:
    res = score.call(model)

    print(score._name)
    print(res)

    if isinstance(score, TopTokenCoherence):
        res_by_topic = score.call_by_topic(model)

        print(res_by_topic)

coherence_20
[1.20549603]
{0: array([0.58744055]), 1: array([1.4888009]), 2: array([1.25667956]), 3: array([0.62309629]), 4: array([0.80754245]), 5: array([1.40719586]), 6: array([1.38072351]), 7: array([0.84087763]), 8: array([0.96687029]), 9: array([1.63864398]), 10: array([0.98878543]), 11: array([1.65296584]), 12: array([1.66568428]), 13: array([1.52890637]), 14: array([1.35914876]), 15: array([0.63813836]), 16: array([1.38530282]), 17: array([1.06580857]), 18: array([2.03071528]), 19: array([0.7965938])}
diversity_euclidean
0.08628544346447767
diversity_jensenshannon
0.08628544346447767
diversity_hellinger
0.08628544346447767
diversity_cosine
0.08628544346447767


In [24]:
model.get_phi(class_ids=MAIN_MODALITY)['topic_18'].sort_values(ascending=False)

modality     token     
@lemmatized  space         0.024810
             launch        0.009205
             nasa          0.007578
             satellite     0.006575
             earth         0.005874
                             ...   
             etal          0.000000
             tacit         0.000000
             sportstalk    0.000000
             nih           0.000000
             hookup        0.000000
Name: topic_18, Length: 52744, dtype: float32

In [25]:
model.class_ids

{'@lemmatized': 1}

In [26]:
KNOWN_METRICS

['euclidean', 'jensenshannon', 'hellinger', 'cosine']

In [21]:
MAIN_MODALITY

'@lemmatized'

In [21]:
def fit_and_compute_scores(model, dataset):
    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    
    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    # print(f'Computing "{coherence_score._name}"...')
    
    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    # print(f'Result by topic: {topic_coherences}.')


    # intra1 = IntratextCoherenceScore(
    #     name='toplen_pwt',
    #     data=dataset,
    #     computation_method=ComputationMethod.SEGMENT_LENGTH,
    #     word_topic_relatedness=WordTopicRelatednessType.PWT,
    #     should_compute=False,  # only on last iter
    # )
    intra2 = IntratextCoherenceScore(
        name='toplen_ptw',
        data=dataset,
        computation_method=ComputationMethod.SEGMENT_LENGTH,
        word_topic_relatedness=WordTopicRelatednessType.PTW,
        should_compute=False,
    )
    # intra3 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     should_compute=False,
    # )
    # intra3_w4 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     window=4,
    #     should_compute=False,
    # )

    intra_topic_coherences = dict()

    for intra in [intra2]:  # [intra2, intra3_w4]:  #[intra1, intra2, intra3]:
        # print(f'\nComputing "{intra._name}"...')

        current_intra_topic_coherences = intra.compute(model)

        assert all(v is not None for v in current_intra_topic_coherences.values())

        _values = current_intra_topic_coherences.values()

        current_intra_topic_coherences = {
            i: current_intra_topic_coherences[t]  # if v is not None else 0.0
            for i, t in enumerate(target_topic_names)
        }

        assert all(abs(x - y) <= 1e-6 for x, y in zip(_values, current_intra_topic_coherences.values())), (_values, current_intra_topic_coherences.values())  # "sorted" Python dicts
        
        intra_topic_coherences[f'topic_coherences_{intra._name}'] = current_intra_topic_coherences

        value = float(np.median(list(current_intra_topic_coherences.values())))
        score_values[intra._name] = value

        # print(f'Result by topic: {current_intra_topic_coherences}.')


    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    
    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
        **intra_topic_coherences,
    }

In [23]:
BEST_PARAMS = dict()

## PLSA

In [79]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [30]:
NUM_TOPICS

20

In [31]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [24]:
BEST_PARAMS[KnownModel.PLSA] = None

## Sparse

In [33]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [34]:
results = dict()

for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['sparse_sp_tau']:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (sparse_sp_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.SPARSE,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={'sparse_sp_tau': sparse_sp_tau, 'smooth_bcg_tau': smooth_bcg_tau}
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
 
            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(-0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
2


...

smooth_phi_bcg: 2.235946795422755
smooth_theta_bcg: 10.435605501971311
sparse_phi_sp: -0.09147055072183996
sparse_theta_sp: -0.4269111341715536
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.235946795422755
smooth_theta_bcg: 10.435605501971311
sparse_phi_sp: -0.09147055072183996
sparse_theta_sp: -0.4269111341715536
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.235946795422755
smooth_theta_bcg: 10.435605501971311
sparse_phi_sp: -0.09147055072183996
sparse_theta_sp: -0.4269111341715536




In [35]:
results

{(-0.05,
  0.05): [{'scores': {'perplexity': 2232.351318359375,
    'coherence_20': array([1.37514454]),
    'diversity_euclidean': 0.06859458065469992,
    'diversity_jensenshannon': 0.6925405437647646,
    'diversity_hellinger': 0.8087537297968894,
    'diversity_cosine': 0.809124514918021},
   'topic_coherences': {0: 1.0510833693989028,
    1: 1.4400805072673324,
    2: 1.405924126520918,
    3: 1.0506564618250258,
    4: 0.9838648633308517,
    5: 1.513185579535105,
    6: 1.376443448129735,
    7: 0.8685955662057809,
    8: 0.6237645959171805,
    9: 0.7415538406306119,
    10: 1.6569088810983559,
    11: 1.3305149261546287,
    12: 1.3272051076959452,
    13: 1.79686216769822,
    14: 1.321134009004414,
    15: 1.4857567187040457,
    16: 1.9358902202133772,
    17: 2.470761747065352,
    18: 1.6914260093876727,
    19: 1.4312786968925868}}, {'scores': {'perplexity': 2236.305908203125,
    'coherence_20': array([1.38645675]),
    'diversity_euclidean': 0.06598491318922854,
    'd

In [40]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

(-0.05, 0.05) 2221.1200358072915
(-0.05, 0.1) 2322.1421712239585
(-0.1, 0.05) 2308.8470865885415
(-0.1, 0.1) 2416.2005208333335


In [41]:
# Best: (-0.05, 0.05) 2221.1200358072915

In [25]:
BEST_PARAMS[KnownModel.SPARSE] = {
    'sparse_sp_tau': -0.05,
    'smooth_bcg_tau': 0.05,
}

In [46]:
model.get_phi()['topic_8'].sort_values(ascending=False)

modality     token    
@lemmatized  armenian     0.019870
             israel       0.012188
             jew          0.011370
             turkish      0.011085
             war          0.009569
                            ...   
             unscuffed    0.000000
             mutlu        0.000000
             brazen       0.000000
             evelope      0.000000
             hookup       0.000000
Name: topic_8, Length: 52744, dtype: float32

## Decorrelation

In [47]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [48]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [49]:
DECORRELATION_TAUS = [0.01] + PARAMS_EXPLORED[KnownModel.DECORRELATION]['decorrelation_tau']

In [50]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (decorrelation_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': decorrelation_tau,
                    'smooth_bcg_tau': smooth_bcg_tau,
                    'sparse_sp_tau': 0.0,
                }
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")

            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(0.01, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
2


...

smooth_phi_bcg: 2.235946795422755
smooth_theta_bcg: 10.435605501971311
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.235946795422755
smooth_theta_bcg: 10.435605501971311
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.235946795422755
smooth_theta_bcg: 10.435605501971311
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1




In [52]:
len(results)

8

In [51]:
results

{(0.01,
  0.05): [{'scores': {'perplexity': 2074.646240234375,
    'coherence_20': array([1.40555233]),
    'diversity_euclidean': 0.06490013388668842,
    'diversity_jensenshannon': 0.6871680885500343,
    'diversity_hellinger': 0.7982264207952608,
    'diversity_cosine': 0.8170687413581699},
   'topic_coherences': {0: 1.108471156085784,
    1: 1.5015728960102719,
    2: 1.43872493324088,
    3: 1.050656461825026,
    4: 1.1310519308541085,
    5: 1.536746809625739,
    6: 1.2859708758937871,
    7: 1.0505878939702333,
    8: 0.6318088871373785,
    9: 0.7955070720815773,
    10: 1.6672885770220354,
    11: 1.376071700260118,
    12: 1.4604697435806584,
    13: 1.33615890556931,
    14: 1.321134009004414,
    15: 1.5812730665298966,
    16: 2.0006336814026606,
    17: 2.637061104654378,
    18: 1.7837763042970165,
    19: 1.416080581385518}}, {'scores': {'perplexity': 2084.033935546875,
    'coherence_20': array([1.30486312]),
    'diversity_euclidean': 0.0627889853151782,
    'divers

In [54]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    print(k, mean_ppl)

(0.01, 0.05) 2062.435506184896
(0.01, 0.1) 2152.06298828125
(0.02, 0.05) 2063.7498372395835
(0.02, 0.1) 2153.3832194010415
(0.05, 0.05) 2081.5203450520835
(0.05, 0.1) 2170.955810546875
(0.1, 0.05) 2175.236328125
(0.1, 0.1) 2248.545654296875


In [ ]:
#  Best: (0.01, 0.05) 2062.435506184896
# Close: (0.02, 0.05) 2063.7498372395835
#        (0.05, 0.05) 2081.5203450520835

In [26]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.01,
    'smooth_bcg_tau': 0.05,
}

## ARTM

In [56]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [57]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [58]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [59]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.ARTM]['sparse_sp_tau']:
        for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
            key = (decorrelation_tau, sparse_sp_tau, smooth_bcg_tau)
            results[key] = []
    
            print(key)
    
            for seed in range(NUM_TRAINS):
                print(seed)
                
                model = init_model_from_family(
                    family=KnownModel.ARTM,
                    dataset=dataset,
                    main_modality=MAIN_MODALITY,
                    num_topics=NUM_TOPICS,
                    seed=seed,
                    model_params={
                        'decorrelation_tau': decorrelation_tau,
                        'smooth_bcg_tau': smooth_bcg_tau,
                        'sparse_sp_tau': sparse_sp_tau,
                    }
                )
    
                for reg in model.regularizers.data:
                    print(f"{reg}: {model.regularizers[reg].tau}")
    
                scores = fit_and_compute_scores(model, dataset)
                results[key].append(scores)

            print()

        print()

    print()

(0.01, -0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
decorrelation: 0.01
2


...

smooth_phi_bcg: 2.235946795422755
smooth_theta_bcg: 10.435605501971311
sparse_phi_sp: -0.09147055072183996
sparse_theta_sp: -0.4269111341715536
decorrelation: 0.1
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.235946795422755
smooth_theta_bcg: 10.435605501971311
sparse_phi_sp: -0.09147055072183996
sparse_theta_sp: -0.4269111341715536
decorrelation: 0.1
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 2.235946795422755
smooth_theta_bcg: 10.435605501971311
sparse_phi_sp: -0.09147055072183996
sparse_theta_sp: -0.4269111341715536
decorrelation: 0.1





In [62]:
len(results)

16

In [65]:
ppls = []

for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    ppls.append(mean_ppl)

    print(k, mean_ppl)

(0.01, -0.05, 0.05) 2226.6844889322915
(0.01, -0.05, 0.1) 2326.734375
(0.01, -0.1, 0.05) 2314.91552734375
(0.01, -0.1, 0.1) 2421.041259765625
(0.02, -0.05, 0.05) 2233.8038736979165
(0.02, -0.05, 0.1) 2333.79833984375
(0.02, -0.1, 0.05) 2321.8982747395835
(0.02, -0.1, 0.1) 2426.70703125
(0.05, -0.05, 0.05) 2262.5601399739585
(0.05, -0.05, 0.1) 2359.8883463541665
(0.05, -0.1, 0.05) 2345.5982259114585
(0.05, -0.1, 0.1) 2448.0912272135415
(0.1, -0.05, 0.05) 2334.3914388020835
(0.1, -0.05, 0.1) 2427.4431966145835
(0.1, -0.1, 0.05) 2411.8780110677085
(0.1, -0.1, 0.1) 2509.6292317708335


In [68]:
sorted(ppls)[:5]

[2226.6844889322915,
 2233.8038736979165,
 2262.5601399739585,
 2314.91552734375,
 2321.8982747395835]

In [ ]:
#  Best: (0.01, -0.05, 0.05) 2226.6844889322915
# Close: (0.02, -0.05, 0.05) 2233.8038736979165

In [27]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.01,
    'sparse_sp_tau':    -0.05,
    'smooth_bcg_tau':    0.05,
}

## TLESS

In [70]:
PARAMS_EXPLORED[KnownModel.TLESS]

{}

In [71]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.TLESS,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [72]:
results

[{'scores': {'perplexity': 2443.865966796875,
   'coherence_20': array([1.30008996]),
   'diversity_euclidean': 0.1154117169554388,
   'diversity_jensenshannon': 0.7912962506978136,
   'diversity_hellinger': 0.9362109719292934,
   'diversity_cosine': 0.9529976794974789},
  'topic_coherences': {0: 0.6166322474785214,
   1: 1.5098768796049284,
   2: 1.6699768743582453,
   3: 1.313431146147788,
   4: 0.8202106988383335,
   5: 1.6633309636689342,
   6: 1.4064045539905468,
   7: 0.8391430204387993,
   8: 0.7431374509565462,
   9: 0.7912218098109653,
   10: 1.042089271987571,
   11: 1.4411138205491438,
   12: 1.2141684377616995,
   13: 1.4150662477034928,
   14: 1.3394558696335013,
   15: 1.3643702101480597,
   16: 1.583211278577641,
   17: 2.213743012885981,
   18: 1.7570446958949038,
   19: 1.2581706384127003}},
 {'scores': {'perplexity': 2417.395263671875,
   'coherence_20': array([1.31074879]),
   'diversity_euclidean': 0.12011498958570174,
   'diversity_jensenshannon': 0.795386100595778

In [73]:
# Best:

In [28]:
BEST_PARAMS[KnownModel.TLESS] = None

## LDA

In [75]:
PARAMS_EXPLORED[KnownModel.LDA]

{'prior': ['symmetric', 'asymmetric', 'heuristic']}

In [76]:
results = dict()

for prior in PARAMS_EXPLORED[KnownModel.LDA]['prior']:
    key = prior
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.LDA,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={'prior': prior}
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        scores = fit_and_compute_scores(model, dataset)
        results[key].append(scores)

    print()

symmetric
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
2


...

smooth_phi: 0.01
smooth_theta: 2.5
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5



In [77]:
results

{'symmetric': [{'scores': {'perplexity': 2075.4833984375,
    'coherence_20': array([1.23178849]),
    'diversity_euclidean': 0.0560873127900243,
    'diversity_jensenshannon': 0.6427780599379403,
    'diversity_hellinger': 0.7184140556339036,
    'diversity_cosine': 0.7756651945925207},
   'topic_coherences': {0: 0.6809874777163635,
    1: 1.0549700556216997,
    2: 1.4175532795308239,
    3: 1.009343537285833,
    4: 0.8200591597718335,
    5: 1.731680379958497,
    6: 1.2033129950712496,
    7: 0.9691342999813533,
    8: 0.6195399396999054,
    9: 0.7016944582492008,
    10: 1.513046712330175,
    11: 1.4763069270094253,
    12: 1.3246030696968405,
    13: 1.2441074862402908,
    14: 1.0658059934606086,
    15: 0.9718967898286901,
    16: 1.9710237921058622,
    17: 2.0660996046986493,
    18: 1.7309365977516677,
    19: 1.0636672547386306}},
  {'scores': {'perplexity': 2050.6669921875,
    'coherence_20': array([1.2325802]),
    'diversity_euclidean': 0.05771187836805078,
    'dive

In [78]:
ppls = []

for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    ppls.append(mean_ppl)

    print(k, mean_ppl)

symmetric 2046.3717854817708
asymmetric 2049.0808919270835
heuristic 2416.7548828125


In [79]:
sorted(ppls)

[2046.3717854817708, 2049.0808919270835, 2416.7548828125]

In [80]:
# Best: symmetric 2046.3717854817708

In [29]:
BEST_PARAMS[KnownModel.LDA] = {
    'prior': 'symmetric',
}

In [82]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [22]:
BEST_PARAMS = {KnownModel.PLSA: None,
 KnownModel.SPARSE: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 KnownModel.DECORRELATION: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 KnownModel.TLESS: None,
 KnownModel.LDA: {'prior': 'symmetric'}}

In [23]:
import json
import warnings

warnings.simplefilter('ignore', UserWarning)

In [24]:
NUM_TRAINS = 20  # 100
COHERENCES = {
    'topic_coherences': list(),
    # 'topic_coherences_toplen_pwt': list(),
    'topic_coherences_toplen_ptw': list(),
    # 'topic_coherences_topden_ptw': list(),
}

In [25]:
SAVE_FOLDER = 'results50_intra/20newsgroups'

! mkdir -p $SAVE_FOLDER

In [26]:
! ls results50_intra

20newsgroups  mkb10  postnauka


## Check if all OK with PPL

In [95]:
import artm

In [98]:
model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=20,
    seed=0,
)

In [99]:
model._model.scores.add(
    artm.scores.PerplexityScore('test_ppl')
)
model._model.scores.add(
    artm.scores.PerplexityScore('test_ppl2', dictionary=dataset.get_dictionary())
)

In [100]:
results = fit_and_compute_scores(model, dataset)

In [101]:
model.scores[f'PerplexityScore{MAIN_MODALITY}']

[50648.61328125,
 4345.74365234375,
 3630.943359375,
 2942.807373046875,
 2561.388427734375,
 2363.885009765625,
 2258.72412109375,
 2195.7939453125,
 2154.23876953125,
 2125.15478515625,
 2103.3349609375,
 2086.39208984375,
 2072.968017578125,
 2061.62939453125,
 2051.577392578125,
 2043.2095947265625,
 2035.985595703125,
 2029.4619140625,
 2024.0592041015625,
 2019.290283203125]

In [102]:
model.scores['test_ppl']

[50648.61328125,
 4345.74365234375,
 3630.943359375,
 2942.807373046875,
 2561.388427734375,
 2363.885009765625,
 2258.72412109375,
 2195.7939453125,
 2154.23876953125,
 2125.15478515625,
 2103.3349609375,
 2086.39208984375,
 2072.968017578125,
 2061.62939453125,
 2051.577392578125,
 2043.2095947265625,
 2035.985595703125,
 2029.4619140625,
 2024.0592041015625,
 2019.290283203125]

In [103]:
model.scores['test_ppl2']

[50648.61328125,
 4345.74365234375,
 3630.943359375,
 2942.807373046875,
 2561.388427734375,
 2363.885009765625,
 2258.72412109375,
 2195.7939453125,
 2154.23876953125,
 2125.15478515625,
 2103.3349609375,
 2086.39208984375,
 2072.968017578125,
 2061.62939453125,
 2051.577392578125,
 2043.2095947265625,
 2035.985595703125,
 2029.4619140625,
 2024.0592041015625,
 2019.290283203125]

In [27]:
# PLSA

start = time.time()

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    # COHERENCES.extend(
    #     list(results['topic_coherences'].values())
    # )

    for k in COHERENCES:
        COHERENCES[k].extend(
            list(results[k].values())
        )

end = time.time()

print(f'Elapsed: {(end - start) / 60} min')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 153.30180006027223 min


In [ ]:
1

In [37]:
scores

[{'perplexity': 1485.8199462890625,
  'coherence_20': array([1.41739557]),
  'diversity_euclidean': 0.0809321941903605,
  'diversity_jensenshannon': 0.7112341804239385,
  'diversity_hellinger': 0.8316939206502058,
  'diversity_cosine': 0.8505134555405262},
 {'perplexity': 1511.845947265625,
  'coherence_20': array([1.38052956]),
  'diversity_euclidean': 0.08058577814891069,
  'diversity_jensenshannon': 0.708602497147086,
  'diversity_hellinger': 0.8280190631845763,
  'diversity_cosine': 0.8477141815137604},
 {'perplexity': 1482.9755859375,
  'coherence_20': array([1.47731498]),
  'diversity_euclidean': 0.082878093807111,
  'diversity_jensenshannon': 0.7119096426146087,
  'diversity_hellinger': 0.8327556919250606,
  'diversity_cosine': 0.8547494793986318},
 {'perplexity': 1492.080078125,
  'coherence_20': array([1.46875148]),
  'diversity_euclidean': 0.08433904853317951,
  'diversity_jensenshannon': 0.7138594510602025,
  'diversity_hellinger': 0.8349923332456672,
  'diversity_cosine': 0

In [28]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [28]:
scores[0]

{'scores': {'perplexity': 1485.8199462890625,
  'coherence_20': 1.4173955734257089,
  'diversity_euclidean': 0.08093218736696144,
  'diversity_jensenshannon': 0.7112341713209872,
  'diversity_hellinger': 0.8316939036124341,
  'diversity_cosine': 0.8505134350332703},
 'topic_coherences': {0: 0.5984603770987272,
  1: 1.041164772835412,
  2: 1.820282555188892,
  3: 1.3378520795347655,
  4: 0.7967952068732984,
  5: 2.608159499624327,
  6: 1.3733187519860877,
  7: 1.3748275754060573,
  8: 0.6234874342413499,
  9: 0.9021065341702647,
  10: 1.1722318995660639,
  11: 1.6287437220444028,
  12: 0.9511380341289424,
  13: 0.8773212497108748,
  14: 1.0746612101074373,
  15: 1.2143488042092232,
  16: 2.224503062835589,
  17: 2.8434886417302216,
  18: 1.1782659384679182,
  19: 1.955132569258584,
  20: 1.0785439449046308,
  21: 1.104910733203592,
  22: 1.3614332397107645,
  23: 0.9243361278635146,
  24: 1.5018185122689924,
  25: 1.2376019370130926,
  26: 1.0740763883785511,
  27: 1.6329172055288808,
 

In [30]:
SAVE_FOLDER

'results50/20newsgroups'

In [29]:
with open(SAVE_FOLDER + '/plsa_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [42]:
len(COHERENCES)

1000

In [43]:
COHERENCES[:10]

[0.5984603770987272,
 1.041164772835412,
 1.820282555188892,
 1.3378520795347655,
 0.7967952068732984,
 2.608159499624327,
 1.3733187519860877,
 1.3748275754060573,
 0.6234874342413499,
 0.9021065341702647]

In [44]:
COHERENCES[:20]

[0.5984603770987272,
 1.041164772835412,
 1.820282555188892,
 1.3378520795347655,
 0.7967952068732984,
 2.608159499624327,
 1.3733187519860877,
 1.3748275754060573,
 0.6234874342413499,
 0.9021065341702647,
 1.1722318995660639,
 1.6287437220444028,
 0.9511380341289424,
 0.8773212497108748,
 1.0746612101074373,
 1.2143488042092232,
 2.224503062835589,
 2.8434886417302216,
 1.1782659384679182,
 1.955132569258584]

In [30]:
# Sparse

start = time.time()

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.SPARSE,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
        model_params=BEST_PARAMS[KnownModel.SPARSE],
    )

    if seed == 0:
        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    # COHERENCES.extend(
    #     list(results['topic_coherences'].values())
    # )

    for k in COHERENCES:
        COHERENCES[k].extend(
            list(results[k].values())
        )

end = time.time()

print(f'Elapsed: {(end - start) / 60} min')

0 smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.019165258246480757
sparse_theta_sp: -0.08944804715975409
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 159.74201964934667 min


In [31]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [32]:
with open(SAVE_FOLDER + '/sparse_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [48]:
len(COHERENCES)

2000

In [49]:
COHERENCES[-10:]

[1.3360419985362597,
 1.474052126818699,
 1.2593367679854806,
 1.3781785201679142,
 1.4907861036605246,
 0.6879135386500477,
 1.1890627845052246,
 2.688999434719961,
 1.3408626728635704,
 1.5564442684400248]

In [50]:
max(COHERENCES)

4.7435245757303495

In [51]:
min(COHERENCES)

0.5197789506047236

In [33]:
def train_many(model_family, save_file_path):
    start = time.time()
    
    scores = []

    # for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
    for seed in range(NUM_TRAINS):
        if seed != NUM_TRAINS - 1:
            print(seed, end=' ')
        else:
            print(seed)
    
        model = init_model_from_family(
            family=model_family,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params=BEST_PARAMS[model_family],
        )
    
        if seed == 0:
            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
    
        results = fit_and_compute_scores(model, dataset)
        # scores.append(results['scores'])
        scores.append(results)
    
        # COHERENCES.extend(
        #     list(results['topic_coherences'].values())
        # )
    
        for k in COHERENCES:
            COHERENCES[k].extend(
                list(results[k].values())
            )
    
    end = time.time()
    
    print(f'Elapsed: {(end - start) / 60} min')

    # for s in scores:
    #     s['coherence_20'] = float(s['coherence_20'])
    
    for s in scores:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

    with open(save_file_path, 'w') as f:
        f.write(
            json.dumps(scores, indent=4)
        )

In [34]:
train_many(KnownModel.DECORRELATION, SAVE_FOLDER + '/decorrelation_with_cohs.json')

0 decorrelation: 0.01
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 156.0125444173813 min


In [54]:
len(COHERENCES)

3000

In [ ]:
1

In [35]:
train_many(KnownModel.TLESS, SAVE_FOLDER + '/tless_with_cohs.json')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 160.00439121325812 min


In [56]:
len(COHERENCES)

4000

In [ ]:
train_many(KnownModel.LDA, SAVE_FOLDER + '/lda_with_cohs.json')

0 smooth_phi: 0.02
smooth_theta: 0.02
1 2 3 4 5 6 7 

In [40]:
len(COHERENCES)

5000

In [72]:
1

1

In [59]:
! ls $SAVE_FOLDER

decorrelation.json  lda.json  plsa.json  sparse.json  tless.json


In [73]:
COHERENCES.keys()

dict_keys(['topic_coherences', 'topic_coherences_toplen_ptw'])

In [74]:
for k in COHERENCES:
    print(k)
    print(len(COHERENCES[k]))

topic_coherences
5000
topic_coherences_toplen_ptw
5000


In [60]:
for p in range(5, 100, 5):
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 5: 0.7575503498812394
10: 0.8809092654318472
15: 0.9517498482895543
20: 1.034617754777075
25: 1.1009404201557875
30: 1.1704562834523393
35: 1.2234188713254668
40: 1.2839383023202575
45: 1.3370523049772112
50: 1.3943024164134359
55: 1.4497828833498478
60: 1.5087886170470994
65: 1.5686230265199284
70: 1.6379660218235124
75: 1.7181516352258615
80: 1.8128360841394622
85: 1.9370390312276458
90: 2.13186919503843
95: 2.4798353302600886


In [75]:
for k in COHERENCES:
    print(k)

    for p in range(5, 100, 5):
        print(f'{p:2}: {np.percentile(COHERENCES[k], p)}')

    print()

topic_coherences
 5: 0.7569290644873368
10: 0.8809092654318472
15: 0.9517498482895543
20: 1.0344350907416338
25: 1.1010082218586374
30: 1.1704562834523393
35: 1.2234188713254668
40: 1.2839383023202575
45: 1.3370523049772112
50: 1.3943024164134359
55: 1.4497828833498478
60: 1.5087886170470994
65: 1.5686230265199284
70: 1.6379660218235124
75: 1.7181516352258615
80: 1.813393592491583
85: 1.9370390312276458
90: 2.13186919503843
95: 2.4798353302600886

topic_coherences_toplen_ptw
 5: 1.3419466978012766
10: 1.3760632784442917
15: 1.402036319564187
20: 1.4261946474701346
25: 1.4485597121950486
30: 1.4727138475208574
35: 1.4976803044331921
40: 1.5218912986847375
45: 1.547377109722304
50: 1.5757406764205872
55: 1.6091601303314262
60: 1.6434974449687076
65: 1.687025481109143
70: 1.7456368947464058
75: 1.8089290964051647
80: 1.891634371846118
85: 2.002975575090499
90: 2.17654750013565
95: 2.576137326741936



In [61]:
min(COHERENCES), max(COHERENCES)

(0.4610162369353376, 4.7435245757303495)

In [62]:
np.argmin(COHERENCES), np.argmax(COHERENCES)

(2299, 1116)

In [76]:
for k in COHERENCES:
    print(k)
    print(min(COHERENCES[k]), max(COHERENCES[k]))
    print()

topic_coherences
0.4610162369353376 4.7435245757303495

topic_coherences_toplen_ptw
1.2157369242052354 5.304383417732438



In [63]:
for p in [2, 98]:
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 2: 0.6623015827448099
98: 2.9735260444237235


In [77]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [78]:
COHERENCES

{'topic_coherences': [0.5984603770987272,
  1.041164772835412,
  1.820282555188892,
  1.3378520795347655,
  0.7967952068732984,
  2.608159499624327,
  1.3733187519860877,
  1.3748275754060573,
  0.6234874342413499,
  0.9021065341702647,
  1.1722318995660639,
  1.6287437220444028,
  0.9511380341289424,
  0.8773212497108748,
  1.0746612101074373,
  1.2143488042092232,
  2.224503062835589,
  2.8434886417302216,
  1.1782659384679182,
  1.955132569258584,
  1.0785439449046308,
  1.104910733203592,
  1.3614332397107645,
  0.9243361278635146,
  1.5018185122689924,
  1.2376019370130926,
  1.0740763883785511,
  1.6329172055288808,
  3.234616508688772,
  0.9372854780553844,
  1.3272460916783875,
  1.3542498191049812,
  1.6145595276059088,
  1.5905279386425104,
  1.9872968853207358,
  1.1334673144776453,
  1.6926105242612695,
  1.387440585739915,
  1.0163306674567487,
  1.2644641264928325,
  1.4972278131954457,
  1.4695714123091284,
  1.8146290663111555,
  1.0262257136046147,
  1.1566272404356803

## Newman Vs. Intratext

### Correlation

In [79]:
COHERENCES.keys()

dict_keys(['topic_coherences', 'topic_coherences_toplen_ptw'])

In [80]:
from scipy.stats import spearmanr

In [81]:
# Before dict filtering
spearmanr(
    COHERENCES['topic_coherences'],
    COHERENCES['topic_coherences_toplen_ptw'],
)

SignificanceResult(statistic=0.20619525139080133, pvalue=3.832155129670343e-49)

### Tops Intersection

In [82]:
inds1 = np.argsort(COHERENCES['topic_coherences'])[::-1][:100]

In [83]:
inds1 = set(np.argsort(COHERENCES['topic_coherences'])[::-1][:100])
inds2 = set(np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:100])

In [84]:
len(inds1 & inds2)

28

### Newman-in-Intratext Density (Mutual Density / Intra-Top Density)

In [85]:
inds2 = np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:100]

In [86]:
np.mean(
    np.array(COHERENCES['topic_coherences'])
)

1.4713587819847318

In [87]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds2]
)

2.4038453469637924

In [88]:
inds2 = np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:200]

In [89]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds2]
)

2.2197421480180117